# 21 · Build a custom sandbox image, then expose it over MCP

Define a sandbox image **in Python**, build/list it, and serve sandbox
operations to an MCP client (Claude Desktop, etc.) as tools.

**Key concepts**
- **`tooling.image.Image`** — a fluent, declarative image spec
  (`dnf_install`, `pip_install`, `run_commands`, `env`, `workdir`); its steps are
  injected into the agent-server Dockerfile and built by `build_image(image=...)`.
- **Snapshot registry** — `list_images()` / `delete_image(arn)` enumerate and
  clean up built images.
- **MCP server** — `RaitboxTools` turns sandbox ops into tools; `build_server()`
  wires them to `FastMCP`, runnable via `python -m raitbox.mcp` so any MCP client
  can create sandboxes and run code.

**Prerequisites**
- AWS creds + region. Building an image also needs an S3 bucket + a build role
  (see `docs/INTEGRATION.md`); that cell is guarded so the notebook runs without.
- Optional: `pip install 'raitbox[mcp]'` for the live `build_server` cell.

In [ ]:
%pip install -q raitbox

## 1 · Describe an image declaratively
This bakes in a language server (for notebook 17) and a desktop stack (for
notebook 18) — no Dockerfile by hand.

In [ ]:
from raitbox.tooling.image import Image
from raitbox.tooling import build_image

# The AWS-managed AL2023 base is aarch64. Desktop tools:
#  - xdotool comes from EPEL 9 (aarch64) — add a repo file pinned to that arch;
#  - screenshots use ImageMagick's `import` (scrot is not packaged for aarch64);
#  - Xvfb is `xorg-x11-server-Xvfb`; xterm is in the base repos.
EPEL = (
    "[epel9]\\nname=EPEL9\\n"
    "baseurl=https://dl.fedoraproject.org/pub/epel/9/Everything/aarch64/\\n"
    "enabled=1\\ngpgcheck=0\\n"
)
img = (
    Image.al2023()
    .dnf_install("git", "xorg-x11-server-Xvfb", "ImageMagick", "xterm")
    .run_commands(f"printf '{EPEL}' > /etc/yum.repos.d/epel9.repo")
    .run_commands("dnf install -y xdotool")
    .pip_install("python-lsp-server", "pandas", "pytest")
    .env("DISPLAY", ":99")
    .workdir("/work")
)
for line in img.dockerfile_lines():
    print(line)

## 2 · See exactly what would be built
`build_image` injects those steps into the base agent-server Dockerfile at a
marker — the server contract (port 8080, lifecycle hooks) is preserved.

In [ ]:
rendered = build_image._render_dockerfile(img)
print(rendered)

## 3 · Build it (guarded) and list the registry
Uncomment to build; it uploads the context to S3, calls `create_microvm_image`,
and polls until the version is ready (minutes).

In [ ]:
import os

BUCKET = os.environ.get("RAITBOX_BUILD_BUCKET")
BUILD_ROLE = os.environ.get("RAITBOX_BUILD_ROLE_ARN")

if BUCKET and BUILD_ROLE:
    arn = build_image.build_image(
        name="raitbox-featured",
        code_bucket=BUCKET,
        build_role_arn=BUILD_ROLE,
        image=img,
    )
    print("built:", arn)
    for entry in build_image.list_images():
        print(entry["state"], entry["name"], entry["arn"])
else:
    print("set RAITBOX_BUILD_BUCKET + RAITBOX_BUILD_ROLE_ARN to build & list images")

## 4 · The MCP tool surface
`RaitboxTools` is the logic behind the MCP server; we drive it directly here (on
`agentcore-ci`, no image needed) to show the tools an MCP client would call.

In [ ]:
from raitbox.mcp import RaitboxTools

tools = RaitboxTools(backend="agentcore-ci")
sid = tools.create_sandbox()
print("created:", sid)
print("run_code:", tools.run_code(sid, "print(6 * 7)"))
tools.files_write(sid, "note.txt", "hi")
print("files   :", tools.files_list(sid))
print("killed  :", tools.kill_sandbox(sid))

## 5 · Wire it into an MCP client
Run the server with `python -m raitbox.mcp` and register it — e.g. in Claude
Desktop's `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "raitbox": {
      "command": "python",
      "args": ["-m", "raitbox.mcp", "--backend", "agentcore-ci"],
      "env": {"AWS_PROFILE": "default", "AWS_DEFAULT_REGION": "us-west-2"}
    }
  }
}
```

In [ ]:
# Optional: build the real FastMCP server object (needs `pip install 'raitbox[mcp]'`).
try:
    from raitbox.mcp import build_server

    server = build_server(name="raitbox", backend="agentcore-ci")
    print("FastMCP server ready:", server.name)
except ImportError:
    print("install the mcp extra to build a live server: pip install 'raitbox[mcp]'")

## Recap
- `Image` described a purpose-built sandbox in Python; `build_image(image=...)`
  builds it and `list_images` / `delete_image` manage the registry.
- `RaitboxTools` / `build_server` expose sandboxes to any MCP client, so an
  assistant can create and drive them as tools.

**Related:** notebooks 17 & 18 (which use images like this one), `docs/CONCEPTS.md`.